In [1]:
# Cell 1: Imports & Setup
from pathlib import Path
import json
import numpy as np
import polars as pl
from sklearn.ensemble import RandomForestClassifier
import gc

data_base_dir = Path('data/regionwise_yearwise_samples')
output_dir = Path('data/feature_selection_artifacts')
output_dir.mkdir(parents=True, exist_ok=True)

TARGET_REGION = 'DES'
REGION_DIR = data_base_dir / TARGET_REGION

# Predictor exclusions: metadata, target, and null-heavy composite bands
cols_to_drop = [
    'class', 'system:index', 'area_m2', 'area_proportion', 'usable_count',
    '.geo', 'q1_count', 'q2_count', 'q3_count', 'q4_count', 'quarters_present',
    'geometry', 'ndvi_q1_median', 'ndvi_q2_median', 'ndvi_q3_median',
    'ndvi_q4_median', 'tir_count', 'tir_mad', 'tir_median', 'snow_count',
    'year'
]

# Standard Level 2 -> Level 1 Remap
old_codes = [5, 3, 4, 12, 11, 66, 19, 36, 9, 41, 24, 23, 75, 30, 25, 33, 76, 34, 61]
new_codes = [5, 3, 4, 12, 11, 66, 19, 14, 14, 14, 22, 22, 22, 22, 22, 33, 76, 34, 61]
class_remap_dict = dict(zip(old_codes, new_codes))

In [2]:
# Cell 2: Scan Partitioned Dataset & Sample 100k Rows
print(f"Scanning partitioned dataset at: {REGION_DIR}...")

# Lazy scan across all yearly parquet partitions
lazy_ds = pl.scan_parquet(REGION_DIR / "**/*.parquet")
schema = lazy_ds.collect_schema()
feature_cols = [c for c in schema.names() if c not in cols_to_drop]

# Remap classes and take a 100k row sample directly out-of-core
df_sample = (
    lazy_ds
    .with_columns(
        pl.col('class').replace_strict(class_remap_dict, default=pl.col('class')).cast(pl.Int32)
    )
    .select(feature_cols + ['class'])
    .collect()
    .sample(n=min(100000, 4500000), seed=42)
)

X_sample = df_sample.select(feature_cols).to_numpy().astype(np.float32)
y_sample = df_sample['class'].to_numpy().astype(np.int32)

del df_sample
gc.collect()

print(f"Sampled baseline array: {X_sample.shape[0]:,} rows, {X_sample.shape[1]} features.")

Scanning partitioned dataset at: data/regionwise_yearwise_samples/DES...
Sampled baseline array: 100,000 rows, 104 features.


In [3]:
# Cell 3: Fit Baseline Random Forest & Export Ranked Features
baseline_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=14,
    n_jobs=4,
    random_state=42
)
baseline_rf.fit(X_sample, y_sample)

feat_importances = dict(zip(feature_cols, [float(v) for v in baseline_rf.feature_importances_]))
all_ranked = sorted(feature_cols, key=lambda c: feat_importances[c], reverse=True)

ranking_artifact = {
    "region": TARGET_REGION,
    "feature_importances": feat_importances,
    "ranked_features": all_ranked
}

ranking_file = output_dir / f"{TARGET_REGION}_ranked_features.json"
with open(ranking_file, "w") as f:
    json.dump(ranking_artifact, f, indent=2)

print(f"Saved ranked features -> {ranking_file}")
print("Top 10 features:", all_ranked[:10])

Saved ranked features -> data/feature_selection_artifacts/DES_ranked_features.json
Top 10 features: ['elevation', 'lat', 'lon', 'slope', 'mndwi_median_dry', 'mndwi_median', 'tcb_median_dry', 'evi2_median_dry', 'hand', 'mndwi_median_wet']
